# Preprocessament dels àudios de lectura

**Objectiu**: Preparar els fitxers d'àudio originals per a l'anàlisi posterior. S'aplica una cadena de processament que inclou reducció de soroll, compressió dinàmica, equalització i retall dels silencis d'inici i final.

**Entrada**: Fitxers `.wav` del directori `raw_data/`, organitzats per escola i classe.

**Sortida**: Fitxers `.wav` processats a `processed_data/`, mantenint l'estructura de directoris original.

**Passos**:
1. Configuració de l'entorn i imports
2. Definició de la cadena de processament d'àudio (`process_sample`)
3. Aplicació del processament per lots a tots els fitxers

## 1. Imports i configuració de l'entorn

Importem les llibreries necessàries:
- `librosa`: càrrega i anàlisi d'àudio
- `noisereduce`: reducció de soroll estacionari
- `pedalboard`: efectes d'àudio (compressor, equalitzador, noise gate)
- `soundfile`: exportació de fitxers `.wav`
- `imageio_ffmpeg`: binari ffmpeg per a la conversió de formats d'àudio

Configurem el directori base de les dades crues (`raw_data/`) i afegim `ffmpeg` al PATH perquè `audioread` el pugui trobar.

In [1]:
from pathlib import Path
import os
import warnings
import numpy as np
import librosa
from pedalboard import *
import noisereduce as nr
import soundfile as sf
import imageio_ffmpeg
os.environ["PATH"] = str(Path(imageio_ffmpeg.get_ffmpeg_exe()).parent) + os.pathsep + os.environ["PATH"]

warnings.filterwarnings("ignore", category=FutureWarning)

BASE_DIR = Path(r"raw_data")
print(f"Directori base: {BASE_DIR.resolve()}")
print(f"Existeix: {BASE_DIR.exists()}")
print(f"ffmpeg: {imageio_ffmpeg.get_ffmpeg_exe()}")

c:\Users\Marçal\OneDrive\Escriptori\TFG\dyslexia-detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Directori base: C:\Users\Marçal\OneDrive\Escriptori\TFG\dyslexia-detection\raw_data
Existeix: True
ffmpeg: c:\Users\Marçal\OneDrive\Escriptori\TFG\dyslexia-detection\.venv\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe


## 2. Definició de la cadena de processament

La funció `process_sample` aplica seqüencialment els passos següents a cada fitxer d'àudio:

1. **Càrrega** de l'àudio a la freqüència de mostreig original
2. **Reducció de soroll estacionari** amb `noisereduce` (intensitat del 80%)
3. **Noise Gate**: silencia els segments per sota de −30 dB per eliminar respiracions i sorolls de fons
4. **Compressor**: redueix la dinàmica per uniformitzar el volum de la veu (threshold −16 dB, ràtio 4:1)
5. **Equalització**: reforça les freqüències de la veu (+10 dB a 400 Hz, +5 dB a 4000 Hz)
6. **Normalització**: escala l'amplitud màxima a 1.0
7. **Retall de silencis**: elimina mostres amb amplitud < 0.01 a l'inici i al final del fitxer

In [2]:
def process_sample(file_path):
    y, sr = librosa.load(file_path, sr=None)
    reduced_noise = nr.reduce_noise(y=y, sr=sr, stationary=True, prop_decrease=0.8)
    board = Pedalboard([
        NoiseGate(threshold_db=-30, ratio=1.5, release_ms=250),
        Compressor(threshold_db=-16, ratio=4),
        LowShelfFilter(cutoff_frequency_hz=400, gain_db=10, q=1),
        HighShelfFilter(cutoff_frequency_hz=4000, gain_db=5, q=1),
        Gain(gain_db=1)
    ])
    effected = board(reduced_noise, sr)
    effected = effected / np.max(np.abs(effected))
    
    # Tallar silencis al principi i al final
    non_silent_indices = np.where(np.abs(effected) > 0.01)[0]
    if len(non_silent_indices) > 0:
        start_idx = non_silent_indices[0]
        end_idx = non_silent_indices[-1] + 1
        effected = effected[start_idx:end_idx]

    return effected, sr

## 3. Processament de totes les mostres

Recorrem recursivament tots els fitxers `.wav` de `raw_data/`, apliquem `process_sample` a cadascun i els guardem a `processed_data/` mantenint la mateixa estructura de directoris (escola → classe → fitxer).

Si la carpeta de destinació no existeix, es crea automàticament.

In [3]:
processed_dir = BASE_DIR.parent / "processed_data"
for root, dirs, files in os.walk(BASE_DIR):
    for file in files:
        if file.endswith(".wav"):
            file_path = Path(root) / file
            print(f"Processing {file_path}...")
            effected, sr = process_sample(file_path)
            
            # Creem la ruta de sortida mantenint l'estructura de directoris
            relative_path = file_path.relative_to(BASE_DIR)
            output_file_path = processed_dir / relative_path
            output_file_path.parent.mkdir(parents=True, exist_ok=True)
            
            # Guardem l'àudio processat
            sf.write(output_file_path, effected, sr)

Processing raw_data\Escola 1\3A\Aiden8 - Los okapis (Castellano) - LlM_ (Castellano).wav...
Processing raw_data\Escola 1\3A\Biel8 - Los okapis (Castellano) - LlM_ (Català).wav...
Processing raw_data\Escola 1\3A\Cala9 - Los okapis (Castellano) - LlM_ (Català).wav...
Processing raw_data\Escola 1\3A\Chloe8 - Los okapis (Castellano) - LlM_ (Castellano).wav...
Processing raw_data\Escola 1\3A\D_Alex9 - Los okapis (Castellano) - LlM_ (Català).wav...
Processing raw_data\Escola 1\3A\D_Arnau8 - Los okapis (Castellano) - LlM_ (Català).wav...
Processing raw_data\Escola 1\3A\D_Berta8 - Los okapis (Castellano) - LlM_ (Castellano).wav...
Processing raw_data\Escola 1\3A\D_Lena8 - Los okapis (Castellano) - LlM_ (Català).wav...
Processing raw_data\Escola 1\3A\Judith9 - Los okapis (Castellano) - LlM_ (Català).wav...
Processing raw_data\Escola 1\3A\Julia9 - Los okapis (Castellano) - LlM_ (Català).wav...
Processing raw_data\Escola 1\3A\JuliaF9 - Los okapis (Castellano) - LlM_ (Català).wav...
Processing raw